# Cointegration Strategy Testing Framework

This notebook tests baskets for cointegration sustainability and mean reversion speed.

## Testing Pipeline:
1. Load price data from parquet files
2. Generate candidate baskets (or load existing)
3. Test initial cointegration
4. Filter by sustainability across time periods
5. Filter by mean reversion speed
6. Output validated baskets for strategy deployment



In [1]:
import sys
from pathlib import Path
import logging
from datetime import datetime, timedelta, timezone

# Find workspace root
current = Path().resolve()
while current.name != 'hku-datawork' and current.parent != current:
    current = current.parent
workspace_root = current if current.name == 'hku-datawork' else Path('/home/leo/GitHub_Clones/hku_stuff/hku-datawork')
sys.path.insert(0, str(workspace_root))

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Import modules
from hypothesis_testing.cointegration.data_loader import load_price_data
from hypothesis_testing.cointegration.basket_generator import generate_baskets_clustering
from hypothesis_testing.cointegration.utils_parallel import test_baskets_cointegration_parallel
from hypothesis_testing.cointegration.filter_sustainability import filter_baskets_sustainability
from hypothesis_testing.cointegration.filter_mean_reversion import filter_baskets_mean_reversion

logger.info("Imports completed")

2025-11-06 11:56:58,744 - __main__ - INFO - Imports completed


## Configuration

Set data paths, timeframes, and filter thresholds.

In [2]:
# Configuration
DATA_DIR = Path("/workspace-hku/hku-data/test_data")  # Use absolute path to actual data location
TIMEFRAME = '15m'
PRICE_TYPE = 'mark'  # 15m only available as 'mark', 1h only as 'index'
SYMBOLS = None  # None = auto-discover all symbols
# Discover available symbols from parquet files in data directory
pattern = f"*_{PRICE_TYPE}_{TIMEFRAME}_*.parquet"
parquet_files = list(DATA_DIR.glob(pattern))
available_symbols = set()
for f in parquet_files:
    # Pattern: perpetual_{SYMBOL}_mark_15m_...
    parts = f.stem.split('_')
    if len(parts) >= 2:
        symbol = parts[1]  # Extract symbol (e.g., "BTC" or "1000BONK")
        available_symbols.add(symbol)

AVAILABLE_SYMBOLS = sorted(list(available_symbols))
print(f"Found {len(AVAILABLE_SYMBOLS)} symbols available in {DATA_DIR}")

# Basket generation
MIN_BASKET_SIZE = 2
MAX_BASKET_SIZE = 4
N_CLUSTERS = 5

# Sustainability filter thresholds
PERSISTENCE_THRESHOLD = 0.7  # 70% of windows/periods must pass
WINDOW_DAYS = 10
STEP_DAYS = 2
PERIOD_DAYS = 10

# Mean reversion filter thresholds
HALF_LIFE_THRESHOLD_DAYS = 5.0  # Maximum half-life in days

# Parallel processing
MAX_WORKERS = None  # None = use CPU count

# Bars per day (depends on timeframe)
BARS_PER_DAY = 96

logger.info(f"Configuration: timeframe={TIMEFRAME}, price_type={PRICE_TYPE}, "
           f"persistence_threshold={PERSISTENCE_THRESHOLD:.0%}, "
           f"half_life_threshold={HALF_LIFE_THRESHOLD_DAYS} days")

2025-11-06 11:57:00,566 - __main__ - INFO - Configuration: timeframe=15m, price_type=mark, persistence_threshold=70%, half_life_threshold=5.0 days


Found 116 symbols available in /workspace-hku/hku-data/test_data


In [3]:
# Load 15-minute mark price data from existing parquet files
# Data is already available in DATA_DIR - no download needed
logger.info(f"Loading data from parquet files in {DATA_DIR}")
logger.info(f"Timeframe: {TIMEFRAME}, Price type: {PRICE_TYPE}")

# Load data using the existing data loader (loads from parquet files)
# SYMBOLS=None means it will discover and load all symbols automatically
price_data = load_price_data(
    data_dir=DATA_DIR,
    years_back=1.2,  # Load only last 2 years
    symbols=None,  # Auto-discover all symbols from parquet files
    timeframe=TIMEFRAME,
    price_type=PRICE_TYPE,
    max_workers=MAX_WORKERS
)

logger.info(f"Loaded price data: {len(price_data)} timestamps, {len(price_data.columns)} symbols")
logger.info(f"Date range: {price_data.index.min()} to {price_data.index.max()}")
print(f"\n✓ Data loaded successfully from parquet files")
print(f"  Symbols: {len(price_data.columns)}")
print(f"  Timestamps: {len(price_data):,}")
print(f"  Date range: {price_data.index.min()} to {price_data.index.max()}")


2025-11-06 11:57:03,325 - __main__ - INFO - Loading data from parquet files in /workspace-hku/hku-data/test_data
2025-11-06 11:57:03,328 - __main__ - INFO - Timeframe: 15m, Price type: mark
2025-11-06 11:57:03,334 - hypothesis_testing.cointegration.data_loader - INFO - Loading data from 116 parquet files for 116 symbols...
2025-11-06 11:57:04,804 - hypothesis_testing.cointegration.data_loader - INFO - Aligning timestamps across all symbols...
2025-11-06 11:57:05,537 - hypothesis_testing.cointegration.data_loader - INFO - Loaded 34939 timestamps for 104 symbols
2025-11-06 11:57:05,539 - hypothesis_testing.cointegration.data_loader - INFO - Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00
2025-11-06 11:57:05,545 - __main__ - INFO - Loaded price data: 34939 timestamps, 104 symbols
2025-11-06 11:57:05,547 - __main__ - INFO - Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00



✓ Data loaded successfully from parquet files
  Symbols: 104
  Timestamps: 34,939
  Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00


## Step 1: Load Price Data

Load historical price data from parquet files.

## Step 2: Generate Candidate Baskets

Generate candidate baskets using clustering (or load existing baskets).

In [4]:
# Extract base symbols (remove _close suffix)
base_symbols = [col.replace('_close', '') for col in price_data.columns]

candidate_baskets = generate_baskets_clustering(
    price_data=price_data,
    n_clusters=N_CLUSTERS,
    min_basket_size=MIN_BASKET_SIZE,
    max_basket_size=MAX_BASKET_SIZE,
    parallel=True,
    max_workers=MAX_WORKERS
)

logger.info(f"Generated {len(candidate_baskets)} candidate baskets")
print(f"Sample baskets: {candidate_baskets[:5]}")

2025-11-06 11:57:10,207 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 69 to 22 symbols (estimated 1001594 -> ~10879 combinations)
2025-11-06 11:57:10,390 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 31 to 22 symbols (estimated 36425 -> ~10879 combinations)
2025-11-06 11:57:10,905 - __main__ - INFO - Generated 18173 candidate baskets


Sample baskets: [['SUN', 'SPELL'], ['GRT', 'GALA'], ['GRT', 'FIL'], ['GRT', 'ARB'], ['GRT', 'NEAR']]


## Step 3: Test Initial Cointegration

Test all candidate baskets for cointegration using Johansen test (multiprocessed).

In [19]:
cointegrated_baskets = test_baskets_cointegration_parallel(
    price_data=price_data,
    candidate_baskets=candidate_baskets,
    max_workers=MAX_WORKERS,
    batch_size=100
)

logger.info(f"Found {len(cointegrated_baskets)} cointegrated baskets out of {len(candidate_baskets)} candidates")
print(f"Cointegration rate: {len(cointegrated_baskets)/len(candidate_baskets):.1%}")

2025-11-06 11:50:45,389 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 10/182 batches (1000 baskets), found 831 cointegrated so far
2025-11-06 11:50:50,983 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 20/182 batches (2000 baskets), found 1790 cointegrated so far
2025-11-06 11:50:55,405 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 30/182 batches (3000 baskets), found 2790 cointegrated so far
2025-11-06 11:50:59,833 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 40/182 batches (4000 baskets), found 3790 cointegrated so far
2025-11-06 11:51:04,249 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 50/182 batches (5000 baskets), found 4790 cointegrated so far
2025-11-06 11:51:08,525 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 60/182 batches (6000 baskets), found 5790 cointegrated so far
2025-11-06 11:51:12,616 - hypothesis_testing.cointegration.utils_parall

Cointegration rate: 97.2%


## Step 4: Filter by Sustainability

Test cointegration persistence across rolling windows and discrete periods. 
This ensures the cointegration relationship is stable over time (not just in-sample).

In [ ]:
sustainable_baskets = filter_baskets_sustainability(
    baskets=cointegrated_baskets,
    price_data=price_data,
    persistence_threshold=PERSISTENCE_THRESHOLD,
    window_days=WINDOW_DAYS,
    step_days=STEP_DAYS,
    period_days=PERIOD_DAYS,
    bars_per_day=BARS_PER_DAY,
    use_rolling=True,
    use_discrete=True,
    max_workers=MAX_WORKERS
)

logger.info(f"Filtered to {len(sustainable_baskets)} sustainable baskets "
           f"(from {len(cointegrated_baskets)} cointegrated)")

# Display sample results
if sustainable_baskets:
    sample = sustainable_baskets[0]
    print(f"\nSample sustainable basket: {sample['basket']}")
    print(f"Rolling windows persistence: {sample['sustainability']['rolling_windows']['persistence_ratio']:.1%}")
    print(f"Discrete periods persistence: {sample['sustainability']['discrete_periods']['persistence_ratio']:.1%}")

2025-11-06 11:52:03,903 - hypothesis_testing.cointegration.filter_sustainability - INFO - Testing 17671 baskets for sustainability...
Exception ignored in: <function _releaseLock at 0x73e552fd8ca0>
Traceback (most recent call last):
  File "/usr/lib/python3.10/logging/__init__.py", line 228, in _releaseLock
    def _releaseLock():
KeyboardInterrupt: 
Process ForkProcess-885:
Process ForkProcess-896:
Process ForkProcess-889:
Process ForkProcess-895:
Process ForkProcess-884:
Process ForkProcess-898:
Process ForkProcess-897:
Process ForkProcess-880:
Process ForkProcess-892:
Process ForkProcess-893:
Process ForkProcess-876:
Process ForkProcess-882:
Process ForkProcess-888:
Process ForkProcess-887:
Process ForkProcess-878:
Process ForkProcess-894:
Process ForkProcess-890:
Process ForkProcess-886:
Process ForkProcess-891:
Process ForkProcess-883:
Process ForkProcess-881:
Process ForkProcess-877:
Traceback (most recent call last):
Process ForkProcess-875:
Traceback (most recent call last):
Tr

## Step 5: Filter by Mean Reversion Speed

Filter for baskets that mean revert quickly (important for trading performance).
This tests whether the spread reverts fast enough to be profitable.

In [ ]:
fast_mean_reverting_baskets = filter_baskets_mean_reversion(
    baskets=sustainable_baskets,
    half_life_threshold_days=HALF_LIFE_THRESHOLD_DAYS,
    bars_per_day=BARS_PER_DAY,
    max_workers=MAX_WORKERS
)

logger.info(f"Filtered to {len(fast_mean_reverting_baskets)} fast mean-reverting baskets "
           f"(from {len(sustainable_baskets)} sustainable)")

# Display sample results
if fast_mean_reverting_baskets:
    sample = fast_mean_reverting_baskets[0]
    print(f"\nSample fast mean-reverting basket: {sample['basket']}")
    print(f"Half-life: {sample['mean_reversion']['half_life_days']:.1f} days")
    print(f"ADF p-value: {sample['mean_reversion']['adf_p_value']:.4f}")
    print(f"Is stationary: {sample['mean_reversion']['is_stationary']}")

## Step 6: Save Validated Baskets

Save the proven baskets for strategy deployment.

In [ ]:
import json

# Prepare baskets for saving (convert numpy arrays to lists)
validated_baskets_output = []
for basket_result in fast_mean_reverting_baskets:
    output = {
        'basket': basket_result['basket'],
        'eigenvector': basket_result['eigenvector'].tolist(),
        'johansen_p_value': basket_result['johansen_result']['p_value'],
        'johansen_trace_stat': float(basket_result['johansen_result']['trace_stat']),
        'sustainability_rolling': basket_result['sustainability']['rolling_windows']['persistence_ratio'],
        'sustainability_discrete': basket_result['sustainability']['discrete_periods']['persistence_ratio'],
        'half_life_days': float(basket_result['mean_reversion']['half_life_days']),
        'adf_p_value': float(basket_result['mean_reversion']['adf_p_value']),
        'is_stationary': basket_result['mean_reversion']['is_stationary']
    }
    validated_baskets_output.append(output)

# Save to JSON
output_file = Path("validated_baskets.json")
with open(output_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'config': {
            'timeframe': TIMEFRAME,
            'price_type': PRICE_TYPE,
            'persistence_threshold': PERSISTENCE_THRESHOLD,
            'half_life_threshold_days': HALF_LIFE_THRESHOLD_DAYS
        },
        'baskets': validated_baskets_output
    }, f, indent=2)

logger.info(f"Saved {len(validated_baskets_output)} validated baskets to {output_file}")

# Display summary
print(f"\n{'='*60}")
print("VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Total candidate baskets: {len(candidate_baskets)}")
print(f"Cointegrated baskets: {len(cointegrated_baskets)} ({len(cointegrated_baskets)/len(candidate_baskets):.1%})")
print(f"Sustainable baskets: {len(sustainable_baskets)} ({len(sustainable_baskets)/len(cointegrated_baskets):.1%} of cointegrated)")
print(f"Fast mean-reverting baskets: {len(fast_mean_reverting_baskets)} ({len(fast_mean_reverting_baskets)/len(sustainable_baskets):.1%} of sustainable)")
print(f"{'='*60}")
print(f"\nValidated baskets ready for strategy deployment!")
print(f"Output file: {output_file}")